# Notebook 4: A Kit of Moves with Nothing Wasted (basis and dimension)

**Predict, then run.** Run the two setup cells first.

<small>Copyright © 2026-present CAehi.us. This whole notebook, text, code and pictures, is licensed under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).</small>

<p><a href="https://colab.research.google.com/github/frcusaca/admath/blob/jia/notebooks/03_getting_back_home_dependence.ipynb" target="_blank" rel="noopener" title="Previous: Notebook 3: Getting Back Home (dependent and independent moves) (opens in a new tab)"><b style="color:#00875f">&lt;&lt;</b> Notebook 3</a></p>

**Setup.** The next cell gets the drawing library, `vecdraw`. The library is short, and you are allowed to read it and change it. To read all of it, run `import inspect` and then `print(inspect.getsource(vecdraw))`.

In [ ]:
# Setup: this cell gets the vecdraw library. You do not need to read it.
REPO_URL = "https://github.com/frcusaca/admath"   # set in course.toml
REPO_REF = "jia"

import importlib.util, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if importlib.util.find_spec("vecdraw") is None:
        if "CHANGE-ME" in REPO_URL:
            raise RuntimeError("The course repository URL is not set yet. Ask your teacher.")
        %pip install --quiet "vecdraw @ git+{REPO_URL}@{REPO_REF}#subdirectory=python"
else:
    # On a computer with the course folder: use the library in python/src.
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "python" / "src" / "vecdraw").is_dir():
            sys.path.insert(0, str(folder / "python" / "src"))
            break
    if importlib.util.find_spec("vecdraw") is None:
        raise RuntimeError("Cannot find vecdraw. Run `make setup` in the course folder, or `pip install -e python`.")

%reload_ext vecdraw._predict

In [ ]:
import numpy as np
import importlib, vecdraw
importlib.reload(vecdraw)      # pick up any changes you make to the library
from vecdraw import *

## Warm-up: same skills, new numbers

On paper first.

1. Are `(1, 2)` and `(2, 4)` dependent or independent?
2. Are `(1, 2)` and `(3, 1)` dependent or independent?
3. Finish the function below so it returns `True` when the spot is `(0, 0)`.

In [ ]:
a = np.array([1, 2])
b = np.array([3, 1])
k = np.array([2, 4])
m = np.array([4, 3])


def is_home(spot):
    return np.allclose(spot, 0)


print(is_home(2 * a - 1 * k))
print(is_home(2 * a - 1 * b))

## Part 1: Too few, too many, just right

Here is a function that counts the ways to reach a target with whole-number weights from -6 to 6. It takes two things as input: a **list of moves** and a **target**.

Read it. `len(moves)` is how many moves are in the list.

In [ ]:
def count_ways(moves, target):
    ways = 0
    if len(moves) == 1:
        for c in range(-6, 7):
            if np.array_equal(c * moves[0], target):
                ways = ways + 1
    if len(moves) == 2:
        for c in range(-6, 7):
            for d in range(-6, 7):
                if np.array_equal(c * moves[0] + d * moves[1], target):
                    ways = ways + 1
    if len(moves) == 3:
        for c in range(-6, 7):
            for d in range(-6, 7):
                for e in range(-6, 7):
                    if np.array_equal(c * moves[0] + d * moves[1] + e * moves[2], target):
                        ways = ways + 1
    return ways

The charger is at `(7, 4)`. Predict the number of ways for each kit of moves. Then run.

In [ ]:
charger = np.array([7, 4])

print("only a:      ", count_ways([a], charger))
print("a and k:     ", count_ways([a, k], charger))
print("a and b:     ", count_ways([a, b], charger))
print("a, b and m:  ", count_ways([a, b, m], charger))

**Talk about it.**

- Which kits cannot reach the charger at all? What is wrong with them?
- Which kit reaches it in many ways? What is wrong with that one?
- Which kit is just right?

**Your turn.** Try three other chargers. Does the "just right" kit always give exactly one way? (If it gives zero, think about Part 6 of Notebook 1.)

## Part 2: The address of a point

With the kit `a`, `b` there is exactly one pair of weights for each spot. Call that pair the spot's **address**.

Searching is slow. NumPy can find the address directly. First put the moves side by side in a table. A table of numbers like this is called a **2D array**.

In [ ]:
kit = np.column_stack([a, b])
print(kit)
print(kit.shape)        # rows, columns

Each **column** of the table is one move. `np.linalg.solve` finds the weights that reach the target.

In [ ]:
print(np.linalg.solve(kit, charger))

Wrap it in a function so it is easy to use again. Then check the answer by rebuilding the trip.

In [ ]:
def address(target, move1, move2):
    kit = np.column_stack([move1, move2])
    return np.linalg.solve(kit, target)


weights = address(charger, a, b)
print(weights)
print(weights[0] * a + weights[1] * b)      # do we get the charger back?

## Part 3: Slanted graph paper

Normal graph paper is built from the moves `(1, 0)` and `(0, 1)`. The moves `a` and `b` build their own slanted graph paper.

Find the charger at `(7, 4)` on the slanted paper. Start at home. How many `a` steps and how many `b` steps? Does it match the address?

In [ ]:
new_grid(8)
draw_skewed_grid(a, b)
draw_vector(a, label="a")
draw_vector(b, label="b")
draw_point(charger, "charger")
show()

**One spot, many addresses.** The charger did not move. Only the kit of moves changed. Predict each address before you run.

In [ ]:
e1 = np.array([1, 0])
e2 = np.array([0, 1])
p = np.array([1, 1])
q = np.array([1, -1])

print("with (1,0) and (0,1): ", address(charger, e1, e2))
print("with a and b:         ", address(charger, a, b))
print("with (1,1) and (1,-1):", address(charger, p, q))

**Your turn.** Draw the slanted paper for `p` and `q` and find the charger on it by counting steps.

In [ ]:
new_grid(8)
draw_skewed_grid(p, q)
draw_vector(p, label="p")
draw_vector(q, label="q")
draw_point(charger, "charger")
show()

## Part 4: When there is no address

What happens if you ask for an address using the broken kit `a` and `k`? Predict.

`try` means "attempt this". If it crashes, Python runs the `except` part instead of stopping.

In [ ]:
try:
    print(address(charger, a, k))
except np.linalg.LinAlgError:
    print("NumPy gave up: these two moves cannot give every spot an address.")

## The names for this

A kit of moves is a **basis** when it is

- big enough to reach everywhere (its span is the whole space), and
- has nothing wasted (the moves are independent).

With a basis, every spot has **exactly one** address.

The number of moves in a basis is called the **dimension**. A line needs 1 move. Flat paper needs 2. The room you are sitting in needs 3.

## Your turn: is it a basis for flat paper?

Decide on paper and say why. Then test with `address` or `count_ways`.

1. `(1, 0)` and `(0, 1)`
2. `(1, 2)` and `(2, 4)`
3. `(1, 1)` and `(1, -1)`
4. `(1, 0)`, `(0, 1)` and `(1, 1)`
5. `(2, 3)` all by itself

In [ ]:
try:
    print(address(np.array([5, 1]), np.array([1, 1]), np.array([1, -1])))
except np.linalg.LinAlgError:
    print("not a basis")

## Check yourself

On paper.

1. Find the address of `(5, 5)` using the basis `a = (1, 2)`, `b = (3, 1)`.
2. Find the address of `(5, 1)` using the basis `(1, 1)`, `(1, -1)`.
3. Give a basis for the line `y = 3x`. What is the dimension of that line?
4. Why can three moves never be a basis for flat paper?
5. A feature vector in your ML class has 4 numbers. How many moves would a basis for those vectors need?

---

# The Deep End

## Deep 1: The dot-product shortcut

For the basis `p = (1, 1)`, `q = (1, -1)` there is a way to get the address with **no solving at all**: just dot products.

Check that it gives the right address for a few spots. Then try the same shortcut with `a` and `b`. It fails. What is different about `p` and `q`? (Notebook 0, Part 5.)

*This leads to: perpendicular bases and projection (Module 12).*

In [ ]:
spot = np.array([5, 1])

shortcut = np.array([np.dot(spot, p) / np.dot(p, p),
                     np.dot(spot, q) / np.dot(q, q)])
print("shortcut:", shortcut)
print("solve:   ", address(spot, p, q))

## Deep 2: The address translator

A friend gives you an address in the `a`, `b` kit: "2 of a and 1 of b". You only speak `p`, `q`. Write a function that translates any `a`, `b` address into a `p`, `q` address.

Test it: translate, rebuild the spot both ways, and make sure it is the same spot.

*This leads to: change of basis, and multiplying matrices (Modules 5 and 8).*

In [ ]:
def translate(ab_address):
    spot = ab_address[0] * a + ab_address[1] * b
    return spot        # finish me: return the p, q address of this spot


print(translate(np.array([2, 1])))

## Deep 3: Pictures are vectors

A tiny 2-by-2 picture is 4 numbers, one brightness per pixel. So it is a vector with 4 numbers.

Here is a basis for all 2-by-2 pictures: four pictures with a single pixel lit. Build a checkerboard as a linear combination of them. Build a picture that gets brighter from left to right.

How many moves would a basis need for the 28-by-28 pictures your ML class uses?

In [ ]:
import matplotlib.pyplot as plt

pixel1 = np.array([[1, 0], [0, 0]])
pixel2 = np.array([[0, 1], [0, 0]])
pixel3 = np.array([[0, 0], [1, 0]])
pixel4 = np.array([[0, 0], [0, 1]])

picture = 1.0 * pixel1 + 0.3 * pixel2 + 0.6 * pixel3 + 0.0 * pixel4    # change the weights
print(picture)

plt.figure(figsize=(2, 2))
plt.imshow(picture, cmap="gray", vmin=0, vmax=1)
plt.axis("off")
plt.show()

## Deep 4: A basis for a tilted floor

In 3D, look at all the spots where `x + y + z = 0`. That is a flat sheet through home.

Find two moves that stay on the sheet and are independent. How do you know they reach **every** spot on the sheet? What is the dimension of the sheet?

Check a few trips: do they stay on the sheet?

In [ ]:
f1 = np.array([1, -1, 0])
f2 = np.array([0, 0, 0])       # change me

spot = 2 * f1 + 3 * f2
print(spot, " x + y + z =", spot.sum())

## Deep 5: Draw the slanted paper yourself

Print the source of `draw_skewed_grid` and read it. Then write your own version, using matplotlib directly, that draws the `a` lines in one color and the `b` lines in another.

In [ ]:
import inspect
print(inspect.getsource(draw_skewed_grid))

## Deep 6: Write the argument

"With a basis, every spot has exactly one address." You have seen it. Now explain why it **must** be true.

Hint: suppose a spot had two different addresses. Subtract one trip from the other. What kind of trip do you get, and why is that impossible for a basis?

<p><a href="https://colab.research.google.com/github/frcusaca/admath/blob/jia/notebooks/03_getting_back_home_dependence.ipynb" target="_blank" rel="noopener" title="Previous: Notebook 3: Getting Back Home (dependent and independent moves) (opens in a new tab)"><b style="color:#00875f">&lt;&lt;</b> Notebook 3</a></p>

<small>Copyright © 2026-present CAehi.us. Licensed under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).</small>